# Incremental File ingestion using COPY INTO





refrence links-

**source_path**  --> /Volumes/workspace/customer360_bronze/raw_files/parquet_source \
**checkpoint_path** --> /Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customers

**schema_path** --> /Volumes/workspace/customer360_bronze/ingestion_metadata/schemas/customers \
**target_table** --> /Volumes/workspace.customer360_bronze.customers_incremental_day09


## step 1--- create new Target Delta table


In [0]:
%sql
CREATE TABLE IF NOT EXISTS 
workspace.customer360_bronze.customers_copy_into (
custId int,
custname STRING,
status STRING,
city STRING,
state STRING
)
USING DELTA;

### Step 2- ingest the Parquet files (customers_01, customers_02, customers_03) to bronze delta table- customers_copy_into


In [0]:
%sql
COPY INTO workspace.customer360_bronze.customers_copy_into
FROM '/Volumes/workspace/customer360_bronze/raw_files/parquet_source/customers_0[1-3]'
FILEFORMAT = PARQUET;

### Step 3- Validate the first Load

In [0]:
%sql
SELECT * FROM workspace.customer360_bronze.customers_copy_into;

### Step 4- Rerun COPY INTO

Rerun Step 2- Ingest the initial Pqrquet file. This step will not write the rows aagin. it would show- 
````
**num_affected_rows	num_inserted_rows	num_skipped_corrupt_files** 
0	                0	                0

````

**Expected**: Still 7 rows.

Why?**_ COPY INTO _**remembers which source files have **already been loaded into that Delta table**.
Unlike Auto Loader, there is no explicit checkpoint directory.




**Idempotency**
An operation is idempotent when rerunning it produces the same intended result rather than accumulating unintended changes.

For example, rerunning COPY INTO against unchanged source files should not append the same files again.


### Step 5- Add 4th file- customers_04

COPY_OPTIONS ('**mergeSchema**' = 'true');  --> **Merges the dataset, add emailid to existing schema**  

COPY_OPTIONS ('**force**' = 'true') --> 
The force=true option tells COPY INTO to reload matching source files even if they've already been loaded into the target Delta table. Since COPY INTO appends records, this can introduce duplicates unless additional deduplication logic is applied.


In [0]:
%sql
COPY INTO workspace.customer360_bronze.customers_copy_into
from '/Volumes/workspace/customer360_bronze/raw_files/parquet_source/customers_04'
fileformat = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
SELECT * FROM workspace.customer360_bronze.customers_copy_into ORDER BY custId;